# Neural Network — From Scratch
> Fully-connected network with backpropagation. No frameworks.

**Architecture:** Input → Hidden (ReLU) → Output (Softmax)  
**Loss:** Cross-Entropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score
np.random.seed(42)

## Activation Functions

In [ ]:
def relu(z):          return np.maximum(0, z)
def relu_grad(z):     return (z > 0).astype(float)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)
def cross_entropy(y_pred, y_true):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-15), axis=1))

## NeuralNetwork Class

In [ ]:
class NeuralNetwork:
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.01):
        # Xavier initialisation
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2/input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2/hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        self.lr = lr
        self.train_losses = []
        self.val_losses   = []

    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = softmax(self.Z2)
        return self.A2

    def backward(self, X, y_true):
        n = len(X)
        dZ2 = self.A2 - y_true                      # ∂L/∂Z2
        dW2 = self.A1.T @ dZ2 / n
        db2 = dZ2.mean(axis=0, keepdims=True)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * relu_grad(self.Z1)
        dW1 = X.T @ dZ1 / n
        db1 = dZ1.mean(axis=0, keepdims=True)
        # Gradient descent
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1

    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch_size=64):
        n = len(X)
        for epoch in range(epochs):
            idx = np.random.permutation(n)
            for i in range(0, n, batch_size):
                batch = idx[i:i+batch_size]
                out = self.forward(X[batch])
                self.backward(X[batch], y[batch])
            # Record losses
            self.train_losses.append(cross_entropy(self.forward(X), y))
            if X_val is not None:
                self.val_losses.append(cross_entropy(self.forward(X_val), y_val))
            if (epoch+1) % 50 == 0:
                print(f"Epoch {epoch+1}/{epochs}  loss={self.train_losses[-1]:.4f}")

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

## Train on Digits Dataset

In [ ]:
digits = load_digits()
X = digits.data / 16.0          # normalise [0,1]
y = digits.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# One-hot encode labels
ohe = OneHotEncoder(sparse_output=False)
y_tr_oh = ohe.fit_transform(y_tr.reshape(-1,1))
y_te_oh = ohe.transform(y_te.reshape(-1,1))

nn = NeuralNetwork(input_dim=64, hidden_dim=128, output_dim=10, lr=0.05)
nn.fit(X_tr, y_tr_oh, X_val=X_te, y_val=y_te_oh, epochs=200, batch_size=64)

y_pred = nn.predict(X_te)
print(f"\nTest Accuracy: {accuracy_score(y_te, y_pred)*100:.2f}%")

## Learning Curves

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(nn.train_losses, label='Train Loss')
plt.plot(nn.val_losses,   label='Val Loss')
plt.xlabel("Epoch"); plt.ylabel("Cross-Entropy Loss")
plt.title("Neural Network – Learning Curves")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## Visualise Predictions

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.flat, range(16)):
    ax.imshow(X_te[idx].reshape(8,8), cmap='gray')
    c = 'green' if y_pred[idx]==y_te[idx] else 'red'
    ax.set_title(f"P:{y_pred[idx]} T:{y_te[idx]}", color=c, fontsize=8)
    ax.axis('off')
plt.suptitle("Predictions (Green=Correct, Red=Wrong)"); plt.tight_layout(); plt.show()